# Análisis Macroeconómico de Colombia

**Autor:** Frederick Salazar Sanchez <br>
**Fecha:** Marzo 2026 <br>
**Descripción:** Análisis descriptivo completo de los indicadores macroeconómicos de Colombia desde 1960 hasta 2024. Incluye PIB, comercio exterior, inflación, mercado laboral, sector externo, finanzas públicas, demografía y desigualdad, con contexto por período presidencial.

## Configuración e importación de librerias

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np

COUNTRY   = 'COLOMBIA'
PATH_DATA = 'data/macro_economics_indicators_2026.csv'
HEIGHT    = 800

# ── Paleta consistente por presidente ────────────────────────────────────────
PRESIDENTES_ORDEN = [
    'ALBERTO LLERAS CAMARGO',
    'GUILLERMO LEÓN VALENCIA',
    'CARLOS LLERAS RESTREPO',
    'MISAEL PASTRANA BORRERO',
    'ALFONSO LÓPEZ MICHELSEN',
    'JULIO CÉSAR TURBAY AYALA',
    'BELISARIO BETANCUR',
    'VIRGILIO BARCO',
    'CÉSAR GAVIRIA',
    'ERNESTO SAMPER',
    'ANDRÉS PASTRANA',
    'ÁLVARO URIBE VÉLEZ',
    'JUAN MANUEL SANTOS',
    'IVÁN DUQUE',
    'GUSTAVO PETRO',
]
_PALETTE = [
    '#4285F4',  # Alberto Lleras Camargo       - Google Blue
    '#EA4335',  # Guillermo Leon Valencia       - Google Red
    '#34A853',  # Carlos Lleras Restrepo        - Google Green
    '#FBBC05',  # Misael Pastrana Borrero        - Google Yellow
    '#9C27B0',  # Alfonso Lopez Michelsen        - Material Purple
    '#00ACC1',  # Julio Cesar Turbay Ayala       - Material Cyan
    '#FF7043',  # Belisario Betancur             - Material Deep Orange
    '#43A047',  # Virgilio Barco                - Material Green 600
    '#1E88E5',  # Cesar Gaviria                 - Material Blue 600
    '#E53935',  # Ernesto Samper                - Material Red 600
    '#8E24AA',  # Andres Pastrana               - Material Purple 600
    '#00897B',  # Alvaro Uribe Velez            - Material Teal 600
    '#F4511E',  # Juan Manuel Santos            - Material Deep Orange 600
    '#039BE5',  # Ivan Duque                    - Material Light Blue 600
    '#7CB342',  # Gustavo Petro                 - Material Light Green 600
]
PRES_COLOR_MAP = {p: c for p, c in zip(PRESIDENTES_ORDEN, _PALETTE)}
COLOR_SEQ = _PALETTE  # backward compat

def add_trend(fig, x, y, degree=2, row=None, col=None):
    y = np.array(y, dtype=float)
    mask = ~np.isnan(y)
    xf, yf = x[mask], y[mask]
    if len(xf) < degree + 1:
        return
    coeffs = np.polyfit(xf, yf, degree)
    trace = go.Scatter(
        x=xf, y=np.polyval(coeffs, xf),
        mode='lines', line=dict(color='red', dash='dash'),
        name='Tendencia', showlegend=False,
    )
    if row is not None:
        fig.add_trace(trace, row=row, col=col)
    else:
        fig.add_trace(trace)


presidentes_lista = [
    {"year": y, "presidente": p}
    for p, (a, b) in [
        ("Alberto Lleras Camargo",    (1960, 1962)),
        ("Guillermo León Valencia",   (1963, 1966)),
        ("Carlos Lleras Restrepo",    (1967, 1970)),
        ("Misael Pastrana Borrero",   (1971, 1974)),
        ("Alfonso López Michelsen",   (1975, 1978)),
        ("Julio César Turbay Ayala",  (1979, 1982)),
        ("Belisario Betancur",        (1983, 1986)),
        ("Virgilio Barco",            (1987, 1990)),
        ("César Gaviria",             (1991, 1994)),
        ("Ernesto Samper",            (1995, 1998)),
        ("Andrés Pastrana",           (1999, 2002)),
        ("Álvaro Uribe Vélez",        (2003, 2010)),
        ("Juan Manuel Santos",        (2011, 2018)),
        ("Iván Duque",               (2019, 2022)),
        ("Gustavo Petro",             (2023, 2024)),
    ]
    for y in range(a, b + 1)
]
df_presidentes = pd.DataFrame(presidentes_lista)
df_presidentes['presidente'] = df_presidentes['presidente'].str.upper()


In [2]:
df_raw = pd.read_csv(PATH_DATA, sep=';', decimal=',')
df = df_raw[df_raw['country_name'] == COUNTRY].copy()

df = pd.merge(df, df_presidentes, on='year', how='left')

df = df[df['year'] <= 2024].reset_index(drop=True)

df['balance_comercial'] = df['exports_of_goods_and_services'] - df['imports_of_goods_and_services']

df = df.sort_values('year').reset_index(drop=True)

print(f"Período: {df['year'].min()} – {df['year'].max()}")
print(f"Registros: {len(df)} | Columnas: {df.shape[1]}")
df.head(3)

Período: 1960 – 2024
Registros: 65 | Columnas: 29


,country_code,region_name,sub_region_name,intermediate_region,country_name,income_group,year,total_gdp,total_gdp_million,gdp_variation,...,external_debt,external_debt_pct_gdp,deuda_publica,ingresos_tributarios,poblacion,gini,life_expectancy_women,life_expectancy_men,presidente,balance_comercial
0,COL,AMERICAS,LATIN AMERICA AND THE CARIBBEAN,SOUTH AMERICA,COLOMBIA,INGRESO MEDIANO ALTO,1960,4.031153e+09,4031.15,0.00,...,0.0,0.0,0.0,0.0,15606209.0,0.0,59.19,55.10,ALBERTO LLERAS CAMARGO,0.93
1,COL,AMERICAS,LATIN AMERICA AND THE CARIBBEAN,SOUTH AMERICA,COLOMBIA,INGRESO MEDIANO ALTO,1961,4.540448e+09,4540.45,5.09,...,0.0,0.0,0.0,0.0,16095203.0,0.0,59.77,55.71,ALBERTO LLERAS CAMARGO,-0.84
2,COL,AMERICAS,LATIN AMERICA AND THE CARIBBEAN,SOUTH AMERICA,COLOMBIA,INGRESO MEDIANO ALTO,1962,4.955544e+09,4955.54,5.41,...,0.0,0.0,0.0,0.0,16599029.0,0.0,60.31,56.31,ALBERTO LLERAS CAMARGO,-0.01


In [3]:
numeric_cols = [
    'total_gdp_million', 'gdp_variation', 'total_gdp_percapita', 'gdp_percapita_variation',
    'exports_of_goods_and_services', 'imports_of_goods_and_services', 'balance_comercial', 'cuenta_corriente',
    'inflation_rate', 'foreign_direct_investment', 'unemployment_rate',
    'international_reserves', 'external_debt', 'external_debt_pct_gdp',
    'deuda_publica', 'ingresos_tributarios',
    'poblacion', 'gini', 'life_expectancy_women', 'life_expectancy_men'
]

# Tabla de estadísticas descriptivas
desc = df[numeric_cols].describe().round(2)
desc.loc['missing'] = df[numeric_cols].isnull().sum()
desc

,total_gdp_million,gdp_variation,total_gdp_percapita,gdp_percapita_variation,exports_of_goods_and_services,imports_of_goods_and_services,balance_comercial,cuenta_corriente,inflation_rate,foreign_direct_investment,unemployment_rate,international_reserves,external_debt,external_debt_pct_gdp,deuda_publica,ingresos_tributarios,poblacion,gini,life_expectancy_women,life_expectancy_men
count,65.00,65.00,65.00,64.00,65.00,65.00,65.00,65.00,65.00,65.00,65.00,6.500000e+01,6.500000e+01,65.00,65.00,65.00,65.00,65.00,63.00,63.00
mean,121096.47,3.91,2812.34,6.05,15.31,17.08,-1.77,-2.13,13.80,2.01,6.04,1.556334e+10,4.280578e+10,28.41,19.39,4.56,33855828.91,23.98,71.73,65.51
std,130490.73,2.71,2570.20,10.77,2.41,3.94,3.47,2.61,9.18,1.77,6.21,1.934456e+10,5.442240e+10,15.03,30.72,6.70,11238651.39,26.97,6.20,5.19
min,4031.15,-7.19,258.30,-23.68,9.98,9.71,-7.66,-7.84,2.02,0.00,0.00,7.712566e+07,0.000000e+00,0.00,0.00,0.00,15606209.00,0.00,59.19,55.10
25%,15341.40,2.50,643.01,-1.81,13.52,13.80,-4.37,-4.20,5.81,0.45,0.00,1.291230e+09,4.123797e+09,22.73,0.00,0.00,23858810.00,0.00,66.97,62.16
50%,58418.99,4.09,1730.39,5.94,15.69,16.75,-1.58,-2.21,10.87,1.60,8.25,7.907620e+09,1.756082e+10,30.05,0.00,0.00,33760571.00,0.00,73.30,64.97
75%,232468.66,5.41,5250.97,15.80,16.77,20.67,0.62,0.00,22.53,3.38,11.06,2.367059e+10,4.703992e+10,38.64,52.54,13.03,43758808.00,53.50,76.92,69.74
max,418818.15,10.80,8279.10,25.86,20.39,27.89,6.85,4.78,33.80,7.03,20.52,6.189797e+10,2.017636e+11,57.65,91.16,17.56,52886363.00,58.50,79.72,73.84
missing,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.000000e+00,0.000000e+00,0.00,0.00,0.00,0.00,0.00,2.00,2.00


## Análisis del PIB

### Evolucuón PIB por año

In [4]:
## PIB total en millones USD y PIB per cápita
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Bar(x=df['year'], y=df['total_gdp_million'],
                     name='PIB total (mill. USD)', marker_color='steelblue', opacity=0.7),
              secondary_y=False)

fig.add_trace(go.Scatter(x=df['year'], y=df['total_gdp_percapita'],
                          name='PIB per cápita (USD)', mode='lines+markers',
                          line=dict(color='darkorange', width=2)),
              secondary_y=True)

add_trend(fig, df['year'].values, df['total_gdp_million'].values, degree=2)

fig.update_layout(title='PIB Total y PIB per Cápita de Colombia (1960–2024)',
                  xaxis_title='Año', height=HEIGHT)
fig.update_yaxes(title_text='PIB Total (millones USD)', secondary_y=False)
fig.update_yaxes(title_text='PIB per Cápita (USD)', secondary_y=True)
fig.show()

### Crecimiento del PIB Anual por periodo presidencial

In [5]:
## PIB total por período presidencial
fig = px.bar(df, x='year', y='total_gdp_million', color='presidente',
             color_discrete_map=PRES_COLOR_MAP,
             category_orders={'presidente': PRESIDENTES_ORDEN},
             labels={'total_gdp_million': 'PIB (millones USD)', 'year': 'Año'})
add_trend(fig, df['year'].values, df['total_gdp_million'].values, degree=2)
fig.update_layout(
    title='PIB de Colombia por Período Presidencial',
    xaxis_title='Año', yaxis_title='PIB (millones USD)', height=HEIGHT,
)
fig.show()


### % Crecimiento PIB Anual

In [6]:
## Crecimiento del PIB (% anual)
fig = go.Figure()
fig.add_trace(go.Bar(
    x=df['year'], y=df['gdp_variation'],
    marker_color=np.where(df['gdp_variation'] >= 0, 'steelblue', 'tomato'),
    name='Crecimiento PIB',
))
add_trend(fig, df['year'].values, df['gdp_variation'].values)
fig.update_layout(
    title='Crecimiento del PIB (% anual) — Colombia',
    xaxis_title='Año', yaxis_title='% anual', height=HEIGHT, showlegend=False,
)
fig.show()


### % Crecimiento de PIB percápita anual

In [7]:
## Crecimiento del PIB per Cápita (% anual)
var_pc = df['gdp_percapita_variation'].fillna(0)
fig = go.Figure()
fig.add_trace(go.Bar(
    x=df['year'], y=var_pc,
    marker_color=np.where(var_pc >= 0, 'mediumseagreen', 'salmon'),
    name='Crecimiento PIB per cápita',
))
add_trend(fig, df['year'].values, var_pc.values)
fig.update_layout(
    title='Crecimiento del PIB per Cápita (% anual) — Colombia',
    xaxis_title='Año', yaxis_title='% anual', height=HEIGHT, showlegend=False,
)
fig.show()


## Análisis Importaciones vs Exportaciones

### Comparación de Exportaciones vs importaciones (% PIB)

In [8]:
## Exportaciones e importaciones como % del PIB
fig = px.bar(df, x='year', y=['exports_of_goods_and_services', 'imports_of_goods_and_services'],
             barmode='group',
             labels={'value': '% del PIB', 'variable': 'Indicador'},
             color_discrete_map={'exports_of_goods_and_services': 'steelblue',
                                 'imports_of_goods_and_services': 'tomato'})

fig.for_each_trace(lambda t: t.update(name='Exportaciones' if 'exports' in t.name else 'Importaciones'))
fig.update_layout(title='Exportaciones e Importaciones de Colombia (% PIB)',
                  xaxis_title='Año', height=HEIGHT)
fig.show()# Comercio Exterior

### Balanza Comercial de Colombia

In [9]:
## Balance Comercial (Exportaciones − Importaciones, % PIB)
colors_bc = np.where(df['balance_comercial'] >= 0, 'mediumseagreen', 'tomato')
fig = go.Figure()
fig.add_trace(go.Bar(
    x=df['year'], y=df['balance_comercial'],
    marker_color=colors_bc, name='Balance comercial',
))
add_trend(fig, df['year'].values, df['balance_comercial'].values)
fig.update_layout(
    title='Balance Comercial (Exportaciones − Importaciones, % PIB) — Colombia',
    xaxis_title='Año', yaxis_title='% PIB', height=HEIGHT, showlegend=False,
)
fig.show()


### Cuenta Corriente % PIB

In [10]:
## Cuenta Corriente (% PIB)
cc = df['cuenta_corriente'].fillna(0)
colors_cc = np.where(cc >= 0, 'steelblue', 'salmon')
fig = go.Figure()
fig.add_trace(go.Bar(
    x=df['year'], y=cc,
    marker_color=colors_cc, name='Cuenta corriente',
))
add_trend(fig, df['year'].values, cc.values)
fig.update_layout(
    title='Cuenta Corriente (% PIB) — Colombia',
    xaxis_title='Año', yaxis_title='% PIB', height=HEIGHT, showlegend=False,
)
fig.show()


### Exportaciones % PIB por periodo presidencial

In [11]:
## Exportaciones (% PIB) por período presidencial
fig = go.Figure()
for pres in PRESIDENTES_ORDEN:
    sub = df[df['presidente'] == pres]
    if sub.empty:
        continue
    fig.add_trace(go.Bar(
        x=sub['year'], y=sub['exports_of_goods_and_services'],
        name=pres, marker_color=PRES_COLOR_MAP[pres],
    ))
add_trend(fig, df['year'].values, df['exports_of_goods_and_services'].values, degree=2)
fig.update_layout(
    title='Exportaciones de Bienes y Servicios (% PIB) por Período Presidencial — Colombia',
    xaxis_title='Año', yaxis_title='% PIB',
    height=HEIGHT, barmode='stack',
)
fig.show()


In [12]:
## Importaciones (% PIB) por período presidencial
fig = go.Figure()
for pres in PRESIDENTES_ORDEN:
    sub = df[df['presidente'] == pres]
    if sub.empty:
        continue
    fig.add_trace(go.Bar(
        x=sub['year'], y=sub['imports_of_goods_and_services'],
        name=pres, marker_color=PRES_COLOR_MAP[pres],
    ))
add_trend(fig, df['year'].values, df['imports_of_goods_and_services'].values, degree=2)
fig.update_layout(
    title='Importaciones de Bienes y Servicios (% PIB) por Período Presidencial — Colombia',
    xaxis_title='Año', yaxis_title='% PIB',
    height=HEIGHT, barmode='stack',
)
fig.show()


In [13]:
## Inflación anual (IPC, %)
fig = go.Figure()
fig.add_trace(go.Bar(
    x=df['year'], y=df['inflation_rate'],
    marker_color='indianred', name='Inflación',
))
#add_trend(fig, df['year'].values, df['inflation_rate'].values, degree=2)
fig.update_layout(
    title='Inflación Anual de Colombia (1960–2024)',
    xaxis_title='Año', yaxis_title='Inflación (%)', height=HEIGHT, showlegend=False,
)
fig.show()


In [14]:
## Inflación por período presidencial
fig = go.Figure()
for pres in PRESIDENTES_ORDEN:
    sub = df[df['presidente'] == pres]
    if sub.empty:
        continue
    fig.add_trace(go.Bar(
        x=sub['year'], y=sub['inflation_rate'],
        name=pres, marker_color=PRES_COLOR_MAP[pres],
    ))
fig.update_layout(
    title='Inflación por Período Presidencial — Colombia',
    xaxis_title='Año', yaxis_title='Inflación (%)',
    height=HEIGHT, barmode='stack',
)
fig.show()


In [15]:
## Tasa de interés del Banco de la República por año y período presidencial
df_tasa_raw = pd.read_csv('data/tasa_interes.csv')
df_tasa_raw['fecha'] = pd.to_datetime(df_tasa_raw['Periodo(MMM DD, AAAA)'], format='%Y/%m/%d')
df_tasa_raw['year'] = df_tasa_raw['fecha'].dt.year

# Promedio anual y valor de cierre (fin de año)
df_tasa_anual = (
    df_tasa_raw[df_tasa_raw['year'] <= 2024]
    .groupby('year')['Tasa de política monetaria']
    .agg(tasa_promedio='mean', tasa_cierre='last')
    .round(2)
    .reset_index()
)

# Unir con presidentes
df_tasa_anual = pd.merge(df_tasa_anual, df_presidentes, on='year', how='left')

fig = go.Figure()

for pres in PRESIDENTES_ORDEN:
    sub = df_tasa_anual[df_tasa_anual['presidente'] == pres]
    if sub.empty:
        continue
    fig.add_trace(go.Bar(
        x=sub['year'], y=sub['tasa_promedio'],
        name=pres, marker_color=PRES_COLOR_MAP[pres],
        legendgroup=pres,
    ))

fig.add_trace(go.Scatter(
    x=df_tasa_anual['year'], y=df_tasa_anual['tasa_cierre'],
    mode='lines+markers',
    name='Tasa cierre año',
    line=dict(color='black', width=1.5, dash='dot'),
    marker=dict(size=5),
))

add_trend(fig, df_tasa_anual['year'].values, df_tasa_anual['tasa_promedio'].values, degree=2)

fig.update_layout(
    title='Tasa de Política Monetaria del Banco de la República (% anual) por Período Presidencial<br>'
          '<sup>Barras = promedio anual | Línea punteada = valor de cierre del año</sup>',
    xaxis_title='Año',
    yaxis_title='Tasa de interés (%)',
    height=HEIGHT,
    barmode='stack',
)
fig.show()


In [16]:
# ── Construcción del dataframe mensual tasa vs inflación ─────────────────────
df_inf_pm = pd.read_csv('data/inflacion.csv')
df_inf_pm['fecha'] = pd.to_datetime(df_inf_pm['Periodo(MMM, AAAA)'], format='%Y/%m/%d')
df_inf_pm['mes'] = df_inf_pm['fecha'].dt.to_period('M')

df_tasa_pm = pd.read_csv('data/tasa_interes.csv')
df_tasa_pm['fecha'] = pd.to_datetime(df_tasa_pm['Periodo(MMM DD, AAAA)'], format='%Y/%m/%d')
df_tasa_pm['mes'] = df_tasa_pm['fecha'].dt.to_period('M')
df_tasa_pm_m = (
    df_tasa_pm.groupby('mes')['Tasa de política monetaria']
    .last().reset_index()
    .rename(columns={'Tasa de política monetaria': 'tasa'})
)

df_pm = pd.merge(
    df_inf_pm[['mes','fecha','Inflación total']].rename(columns={'Inflación total':'inflacion'}),
    df_tasa_pm_m, on='mes', how='inner'
)
df_pm = df_pm[df_pm['fecha'].dt.year >= 2000].sort_values('fecha').reset_index(drop=True)

import numpy as np
import plotly.graph_objects as go

# --- 1. Preparación de las "Capas" de color ---
# Creamos dos series temporales sintéticas para el sombreado
# 'tasa_restrictiva' solo tendrá valores cuando la tasa sea mayor que la inflación
tasa_restrictiva = np.where(df_pm["tasa"] >= df_pm["inflacion"], df_pm["tasa"], df_pm["inflacion"])

# 'tasa_expansiva' solo tendrá valores cuando la tasa sea menor que la inflación
tasa_expansiva = np.where(df_pm["tasa"] < df_pm["inflacion"], df_pm["tasa"], df_pm["inflacion"])

fig = go.Figure()

# --- 2. Dibujamos la Inflación como línea base de referencia ---
# (La necesitamos primero para que el 'tonexty' sepa contra qué comparar)
fig.add_trace(go.Scatter(
    x=df_pm["fecha"], y=df_pm["inflacion"],
    line=dict(width=0), # Línea invisible, solo sirve de base para el relleno
    showlegend=False,
    hoverinfo='skip'
))

# --- 3. Sombreado VERDE: Tasa > Inflación (Política Contractiva) ---
fig.add_trace(go.Scatter(
    x=df_pm["fecha"], y=tasa_restrictiva,
    fill='tonexty', 
    fillcolor='rgba(214, 39, 40, 0.25)', # Rojo: tasa frena la economía
    line=dict(width=0),
    name="Tasa > Inflación (freno, tasa real positiva)",
    hoverinfo='skip'
))

# --- 4. Sombreado ROJO: Tasa < Inflación (Política Expansiva / Rezago) ---
# Primero reseteamos la base volviendo a dibujar la inflación
fig.add_trace(go.Scatter(
    x=df_pm["fecha"], y=df_pm["inflacion"],
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=df_pm["fecha"], y=tasa_expansiva,
    fill='tonexty',
    fillcolor='rgba(0, 180, 80, 0.25)', # Verde: tasa acelera la economía
    line=dict(width=0),
    name="Inflación > Tasa (acelerador, tasa real negativa)",
    hoverinfo='skip'
))

# --- 5. Dibujamos las líneas principales ENCIMA de los colores ---
fig.add_trace(go.Scatter(
    x=df_pm["fecha"], y=df_pm["tasa"],
    mode="lines",
    name="Tasa BanRep",
    line=dict(color="#1f77b4", width=3), # Azul
))

fig.add_trace(go.Scatter(
    x=df_pm["fecha"], y=df_pm["inflacion"],
    mode="lines",
    name="Inflación anual IPC",
    line=dict(color="#d62728", width=3), # Rojo
))

# --- 6. Formato de Layout ---
fig.update_layout(
    title="<b>Análisis de Política Monetaria: Spread BanRep vs Inflación</b>",
    xaxis_title="Año",
    yaxis_title="% Anual",
    template="plotly_white",
    hovermode="x unified",
    height=HEIGHT,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    shapes=[
        # Una línea en 0 por si las moscas
        dict(type="line", x0=df_pm["fecha"].min(), x1=df_pm["fecha"].max(), y0=0, y1=0, line=dict(color="black", width=1))
    ]
)

fig.show()

In [17]:
import plotly.graph_objects as go

# 1. Calculamos el diferencial (Spread)
df_pm["spread"] = df_pm["tasa"] - df_pm["inflacion"]

# 2. Colores: Verde (Freno) y Rojo (Acelerador)
colors = ['rgba(214, 39, 40, 0.8)' if x >= 0 else 'rgba(0, 160, 80, 0.8)' for x in df_pm["spread"]]

fig = go.Figure()

# 3. Gráfica de barras
fig.add_trace(go.Bar(
    x=df_pm["fecha"],
    y=df_pm["spread"],
    marker_color=colors,
    name="Diferencial",
    hovertemplate="<b>Fecha:</b> %{x}<br><b>Diferencial:</b> %{y:.2f} pp<extra></extra>"
))

# 4. Línea de equilibrio en cero
fig.add_hline(y=0, line_dash="solid", line_color="black", line_width=2)

# 5. TEXTOS EXPLICATIVOS (Corregidos con 'borderpad')
# Texto para la parte superior (Verde)
fig.add_annotation(
    x=0.02, y=0.90, xref="paper", yref="paper",
    text="<b>ZONA VERDE (Arriba de 0):</b><br>El Banco está 'FRENANDO' la economía.<br>• Los créditos son más caros.<br>• Ahorrar en CDTs es muy rentable.",
    showarrow=False, align="left",
    bgcolor="rgba(255, 255, 255, 0.9)", bordercolor="green", borderwidth=2, borderpad=10
)

# Texto para la parte inferior (Roja)
fig.add_annotation(
    x=0.02, y=0.10, xref="paper", yref="paper",
    text="<b>ZONA ROJA (Abajo de 0):</b><br>El Banco está 'ACELERANDO' la economía.<br>• El dinero es 'barato'.<br>• Se incentiva el consumo y la inversión.",
    showarrow=False, align="left",
    bgcolor="rgba(255, 255, 255, 0.9)", bordercolor="red", borderwidth=2, borderpad=10
)

# 6. Configuración del Layout (Altura 800)
fig.update_layout(
    title="<b>¿Qué tan 'Caro' está el dinero en Colombia?</b><br><sup>Diferencial entre Tasa del Banco de la República e Inflación (2000-2026)</sup>",
    xaxis_title="Año",
    yaxis_title="Diferencial (Puntos Porcentuales)",
    template="plotly_white",
    height=800, 
    hovermode="x",
    showlegend=False
)

fig.show()

In [18]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Preparación de datos
df_pm["spread"] = df_pm["tasa"] - df_pm["inflacion"]
df_pm["fecha_label"] = df_pm["fecha"].dt.strftime("%Y-%m")
colors = ['rgba(214, 39, 40, 0.8)' if x >= 0 else 'rgba(0, 160, 80, 0.8)' for x in df_pm["spread"]]

# 2. Crear subplots (2 filas, 1 columna)
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=(
        "1. EL PROBLEMA: Comportamiento de la Inflación",
        "2. LA DECISIÓN: ¿Qué tan fuerte pisa el freno el Banco?"
    )
)

# --- FILA 1: Inflación ---
fig.add_trace(go.Scatter(
    x=df_pm["fecha_label"], y=df_pm["inflacion"],
    mode="lines", name="Inflación",
    line=dict(color="#d62728", width=3),
    hovertemplate="<b>%{x}</b><br>Inflación: %{y:.2f}%<extra></extra>"
), row=1, col=1)

fig.add_hline(y=3, line_dash="dot", line_color="black", opacity=0.5,
              annotation_text="Meta ideal (3%)", row=1, col=1)

# --- FILA 2: Spread (El Freno/Acelerador) ---
fig.add_trace(go.Bar(
    x=df_pm["fecha_label"], y=df_pm["spread"],
    marker_color=colors, name="Diferencial",
    hovertemplate="<b>%{x}</b><br>Spread: %{y:.2f} pp<extra></extra>"
), row=2, col=1)

fig.add_hline(y=0, line_dash="solid", line_color="black", line_width=2, row=2, col=1)

# 3. Textos explicativos
fig.add_annotation(
    x=0.01, y=0.45, xref="paper", yref="paper",
    text="<b>¿Cómo leer esto?</b><br>Si la inflación (arriba) sube,<br>el Banco debe frenar (barras rojas abajo).<br>Si la inflación baja, el Banco debe soltar el freno.",
    showarrow=False, align="left", bgcolor="white", bordercolor="black", borderpad=10
)

fig.add_annotation(
    x=df_pm["fecha_label"].iloc[-1], y=df_pm["spread"].iloc[-1],
    text="<b>Freno Máximo:</b><br>La inflación ya bajó,<br>pero el freno sigue puesto.",
    showarrow=True, arrowhead=2, ax=-80, ay=-50, row=2, col=1
)

# 4. Ajustes finales
fig.update_layout(
    title="<b>Análisis de Decisiones del Banco de la República</b>",
    height=1000,
    template="plotly_white",
    showlegend=False,
)

# Mostrar solo cada 12 meses en el eje x para no saturar
tickvals = df_pm["fecha_label"][::12].tolist()
fig.update_xaxes(
    tickvals=tickvals,
    ticktext=tickvals,
    tickangle=45,
    row=2, col=1
)

fig.update_yaxes(title_text="Inflación (%)", row=1, col=1)
fig.update_yaxes(title_text="Freno (+) / Acelerador (-)", row=2, col=1)

fig.show()

In [19]:
## Análisis bivariable: Inflación anual vs Tasa de interés (2002–2024)

# Preparar datos anuales de tasa de interés
df_tasa_merge = df_tasa_anual[['year', 'tasa_promedio']].rename(
    columns={'tasa_promedio': 'tasa_interes'}
)

# Unir con df principal (inflación anual)
df_biv = pd.merge(
    df[['year', 'inflation_rate', 'presidente']],
    df_tasa_merge,
    on='year', how='inner'
)
df_biv = df_biv[df_biv['year'] >= 2002].copy()

fig = go.Figure()

# Scatter coloreado por presidente
for pres in PRESIDENTES_ORDEN:
    sub = df_biv[df_biv['presidente'] == pres]
    if sub.empty:
        continue
    fig.add_trace(go.Scatter(
        x=sub['tasa_interes'],
        y=sub['inflation_rate'],
        mode='markers+text',
        name=pres,
        marker=dict(color=PRES_COLOR_MAP[pres], size=12,
                    line=dict(width=1, color='white')),
        text=sub['year'].astype(str),
        textposition='top center',
        textfont=dict(size=9),
    ))

# Línea de tendencia global
x_all = df_biv['tasa_interes'].values
y_all = df_biv['inflation_rate'].values
coeffs = np.polyfit(x_all, y_all, 1)
x_line = np.linspace(x_all.min(), x_all.max(), 100)
r = np.corrcoef(x_all, y_all)[0, 1]
fig.add_trace(go.Scatter(
    x=x_line, y=np.polyval(coeffs, x_line),
    mode='lines',
    name=f'Tendencia lineal (r = {r:.2f})',
    line=dict(color='black', dash='dash', width=1.5),
))

# Línea de referencia diagonal (tasa = inflación)
lim = max(x_all.max(), y_all.max()) * 1.05
fig.add_trace(go.Scatter(
    x=[0, lim], y=[0, lim],
    mode='lines',
    name='Tasa = Inflación',
    line=dict(color='gray', dash='dot', width=1),
))

fig.update_layout(
    title='Inflación Anual vs Tasa de Política Monetaria — Colombia (2002–2024)<br>'
          '<sup>Cada punto = un año | La línea gris indica cuando tasa = inflación (tasa real = 0%)</sup>',
    xaxis_title='Tasa de interés promedio anual (%)',
    yaxis_title='Inflación anual (%)',
    height=HEIGHT,
    legend=dict(font=dict(size=9)),
)
fig.show()


# Mercado Laboral

In [20]:
## Tasa de desempleo por período presidencial
fig = px.bar(df, x='year', y='unemployment_rate', color='presidente',
             color_discrete_map=PRES_COLOR_MAP,
             category_orders={'presidente': PRESIDENTES_ORDEN},
             labels={'unemployment_rate': 'Desempleo (%)', 'year': 'Año'})
add_trend(fig, df['year'].values, df['unemployment_rate'].values, degree=2)
fig.update_layout(
    title='Tasa de Desempleo de Colombia por Período Presidencial',
    height=HEIGHT,
)
fig.show()


# Inversión Extranjera y Sector Externo

In [21]:
## Inversión Extranjera Directa Neta (% PIB) por período presidencial
df_ied = pd.merge(df[['year','foreign_direct_investment']], df_presidentes, on='year', how='left')

fig = go.Figure()

for pres in PRESIDENTES_ORDEN:
    sub = df_ied[df_ied['presidente'] == pres]
    if sub.empty:
        continue
    fig.add_trace(go.Bar(
        x=sub['year'], y=sub['foreign_direct_investment'],
        name=pres,
        marker_color=PRES_COLOR_MAP[pres],
        legendgroup=pres,
    ))

add_trend(fig, df_ied['year'].values, df_ied['foreign_direct_investment'].values)

fig.update_layout(
    title='Inversión Extranjera Directa Neta (% PIB) — Colombia por Período Presidencial',
    xaxis_title='Año',
    yaxis_title='% PIB',
    height=HEIGHT,
    barmode='stack',
    legend=dict(orientation='v', yanchor='middle', y=0.5, xanchor='left', x=1.01),
)
fig.show()

In [22]:
## Reservas Internacionales (USD corrientes)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df['year'], y=df['international_reserves'],
    mode='lines+markers', name='Reservas',
    line=dict(color='darkorange', width=2),
    fill='tozeroy', fillcolor='rgba(255,165,0,0.1)',
))
add_trend(fig, df['year'].values, df['international_reserves'].values, degree=2)
fig.update_layout(
    title='Reservas Internacionales (USD corrientes) — Colombia',
    xaxis_title='Año', yaxis_title='USD', height=HEIGHT, showlegend=False,
)
fig.show()


In [23]:
## Reservas Internacionales por período presidencial (USD corrientes)
fig = go.Figure()

for pres in PRESIDENTES_ORDEN:
    sub = df[df['presidente'] == pres]
    if sub.empty:
        continue
    fig.add_trace(go.Bar(
        x=sub['year'],
        y=sub['international_reserves'],
        name=pres,
        marker_color=PRES_COLOR_MAP[pres],
    ))

add_trend(fig, df['year'].values, df['international_reserves'].values, degree=2)

fig.update_layout(
    title='Reservas Internacionales por Período Presidencial — Colombia (USD corrientes)',
    xaxis_title='Año',
    yaxis_title='USD',
    height=HEIGHT,
    barmode='stack',
)
fig.show()


In [24]:
## Deuda externa: stock total y como % del PIB
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Bar(x=df['year'], y=df['external_debt'],
                     name='Deuda externa (USD)', marker_color='steelblue', opacity=0.7),
              secondary_y=False)

fig.add_trace(go.Scatter(x=df['year'], y=df['external_debt_pct_gdp'],
                          name='Deuda externa (% PIB)', mode='lines+markers',
                          line=dict(color='crimson', width=2)),
              secondary_y=True)

add_trend(fig, df['year'].values, df['external_debt_pct_gdp'].values, degree=2)

fig.update_layout(title='Deuda Externa de Colombia: Valor Absoluto y % del PIB',
                  xaxis_title='Año', height=HEIGHT)
fig.update_yaxes(title_text='Deuda Externa (USD)', secondary_y=False)
fig.update_yaxes(title_text='Deuda Externa (% PIB)', secondary_y=True)
fig.show()

In [25]:
## Deuda pública del Gobierno Central (% PIB)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df['year'], y=df['deuda_publica'],
    mode='lines+markers', name='Deuda pública',
    line=dict(color='firebrick', width=2),
    fill='tozeroy', fillcolor='rgba(178,34,34,0.15)',
))
add_trend(fig, df['year'].values, df['deuda_publica'].replace(0, np.nan).values)
fig.update_layout(
    title='Deuda Pública del Gobierno Central (% PIB) — Colombia',
    xaxis_title='Año', yaxis_title='% PIB', height=HEIGHT, showlegend=False,
)
fig.show()


In [26]:
## Ingresos tributarios (% PIB)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df['year'], y=df['ingresos_tributarios'],
    mode='lines+markers', name='Ingresos tributarios',
    line=dict(color='darkgreen', width=2),
    fill='tozeroy', fillcolor='rgba(0,100,0,0.15)',
))
add_trend(fig, df['year'].values, df['ingresos_tributarios'].replace(0, np.nan).values)
fig.update_layout(
    title='Ingresos Tributarios (% PIB) — Colombia',
    xaxis_title='Año', yaxis_title='% PIB', height=HEIGHT, showlegend=False,
)
fig.show()


In [27]:
## Población total
df['poblacion_millones'] = df['poblacion'] / 1_000_000

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['year'], y=df['poblacion_millones'],
                          mode='lines+markers', name='Población',
                          line=dict(color='teal', width=2),
                          fill='tozeroy', fillcolor='rgba(0,128,128,0.1)'))
add_trend(fig, df['year'].values, df['poblacion_millones'].values, degree=2)

fig.update_layout(title='Población Total de Colombia (millones de personas)',
                  xaxis_title='Año', yaxis_title='Millones de personas', height=HEIGHT)
fig.show()# Demografía y Bienestar Social

In [28]:
## Esperanza de vida: mujeres vs hombres
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['year'], y=df['life_expectancy_women'],
                          mode='lines+markers', name='Mujeres',
                          line=dict(color='deeppink', width=2)))
fig.add_trace(go.Scatter(x=df['year'], y=df['life_expectancy_men'],
                          mode='lines+markers', name='Hombres',
                          line=dict(color='royalblue', width=2)))

# Brecha como área sombreada
fig.add_trace(go.Scatter(
    x=pd.concat([df['year'], df['year'][::-1]]),
    y=pd.concat([df['life_expectancy_women'], df['life_expectancy_men'][::-1]]),
    fill='toself', fillcolor='rgba(128,128,128,0.15)',
    line=dict(color='rgba(255,255,255,0)'), name='Brecha', showlegend=True
))

fig.update_layout(title='Esperanza de Vida de Colombia — Mujeres vs Hombres',
                  xaxis_title='Año', yaxis_title='Años', height=HEIGHT)
fig.show()

In [29]:
## Índice de Gini (desigualdad) por período presidencial
df_gini = df[df['gini'] > 0].copy()

fig = px.bar(df_gini, x='year', y='gini', color='presidente',
             color_discrete_map=PRES_COLOR_MAP,
             category_orders={'presidente': PRESIDENTES_ORDEN},
             labels={'gini': 'Índice de Gini', 'year': 'Año'})
add_trend(fig, df_gini['year'].values, df_gini['gini'].values, degree=1)
fig.add_hline(y=50, line_dash='dot', line_color='gray',
              annotation_text='Umbral alto desigualdad (50)',
              annotation_position='top left')
fig.update_layout(
    title='Índice de Gini de Colombia por Período Presidencial',
    yaxis_title='Gini (0 = igualdad perfecta | 100 = máx. desigualdad)',
    height=HEIGHT,
)
fig.show()


In [30]:
## Resumen estadístico por presidente
agg_cols = {
    'total_gdp_million':              'mean',
    'gdp_variation':                  'mean',
    'total_gdp_percapita':            'mean',
    'exports_of_goods_and_services':  'mean',
    'imports_of_goods_and_services':  'mean',
    'balance_comercial':              'mean',
    'inflation_rate':                 'mean',
    'unemployment_rate':              'mean',
    'foreign_direct_investment':      'mean',
    'external_debt_pct_gdp':          'mean',
    'deuda_publica':                  'mean',
    'ingresos_tributarios':           'mean',
    'gini':                           'mean',
    'life_expectancy_women':          'mean',
    'life_expectancy_men':            'mean',
}

df_pres = (df.groupby('presidente', sort=False)
             .agg(agg_cols)
             .round(2)
             .reset_index())

# Añadir período de mandato
mandato = df.groupby('presidente', sort=False)['year'].agg(['min','max']).reset_index()
mandato['mandato'] = mandato['min'].astype(str) + '–' + mandato['max'].astype(str)
df_pres = pd.merge(df_pres, mandato[['presidente','mandato']], on='presidente')

cols_show = ['presidente', 'mandato', 'total_gdp_million', 'gdp_variation',
             'inflation_rate', 'unemployment_rate', 'balance_comercial',
             'foreign_direct_investment', 'external_debt_pct_gdp',
             'deuda_publica', 'ingresos_tributarios', 'gini']
df_pres[cols_show].rename(columns={
    'total_gdp_million': 'PIB (mill USD)',
    'gdp_variation': 'Crec. PIB %',
    'inflation_rate': 'Inflación %',
    'unemployment_rate': 'Desempleo %',
    'balance_comercial': 'Balance Com.',
    'foreign_direct_investment': 'IED % PIB',
    'external_debt_pct_gdp': 'Deuda Ext. %PIB',
    'deuda_publica': 'Deuda Púb. %PIB',
    'ingresos_tributarios': 'Ing. Trib. %PIB',
    'gini': 'Gini'
})

,presidente,mandato,PIB (mill USD),Crec. PIB %,Inflación %,Desempleo %,Balance Com.,IED % PIB,Deuda Ext. %PIB,Deuda Púb. %PIB,Ing. Trib. %PIB,Gini
0,ALBERTO LLERAS CAMARGO,1960–1962,4509.05,3.50,6.26,0.00,0.03,0.00,0.00,0.00,0.00,0.00
1,GUILLERMO LEÓN VALENCIA,1963–1966,5499.70,4.58,16.94,0.00,-0.75,0.00,0.00,0.00,0.00,0.00
2,CARLOS LLERAS RESTREPO,1967–1970,6358.48,5.59,7.42,0.00,-0.48,0.15,8.12,0.00,0.00,0.00
3,MISAEL PASTRANA BORRERO,1971–1974,9794.38,6.52,17.70,0.00,-0.61,0.33,32.14,0.00,0.00,0.00
4,ALFONSO LÓPEZ MICHELSEN,1975–1978,17793.62,4.92,23.71,0.00,2.86,0.31,26.63,0.00,0.00,0.00
5,JULIO CÉSAR TURBAY AYALA,1979–1982,34174.39,3.18,25.71,0.00,-1.36,0.65,23.64,0.00,0.00,13.78
6,BELISARIO BETANCUR,1983–1986,36704.96,3.46,19.67,0.00,1.19,2.00,37.06,0.00,0.00,0.00
7,VIRGILIO BARCO,1987–1990,40742.51,4.28,26.61,0.00,3.39,0.98,42.72,0.00,0.00,26.48
8,CÉSAR GAVIRIA,1991–1994,63936.22,4.31,25.67,8.90,-1.91,1.35,30.39,0.00,0.00,25.65
9,ERNESTO SAMPER,1995–1998,98692.66,2.82,19.71,11.92,-5.97,3.08,30.42,1.35,3.44,14.20


In [31]:
## KPIs promedio por período presidencial — heatmap comparativo
from sklearn.preprocessing import MinMaxScaler

kpi_cols = {
    'gdp_variation':               'Crec. PIB (%)',
    'inflation_rate':              'Inflación (%)',
    'unemployment_rate':           'Desempleo (%)',
    'foreign_direct_investment':   'IED (% PIB)',
    'exports_of_goods_and_services': 'Exportac. (% PIB)',
    'balance_comercial':           'Bal. Comercial',
    'external_debt_pct_gdp':       'Deuda Ext. (% PIB)',
}

df_kpi = df.copy()
for c in kpi_cols:
    df_kpi[c] = df_kpi[c].replace(0, np.nan)

resumen = (
    df_kpi.groupby('presidente')[list(kpi_cols.keys())]
    .mean().round(2)
    .reindex(PRESIDENTES_ORDEN)
    .dropna(how='all')
    .rename(columns=kpi_cols)
)

scaler = MinMaxScaler()
resumen_norm = pd.DataFrame(
    scaler.fit_transform(resumen.fillna(resumen.mean())),
    index=resumen.index, columns=resumen.columns,
)

fig = go.Figure(go.Heatmap(
    z=resumen_norm.values,
    x=resumen_norm.columns.tolist(),
    y=resumen_norm.index.tolist(),
    colorscale='RdYlGn',
    zmid=0.5,
    text=resumen.fillna('—').astype(str).values,
    texttemplate='%{text}',
    textfont=dict(size=9),
    showscale=True,
    colorbar=dict(title='Valor<br>normalizado'),
))
fig.update_layout(
    title='KPIs Promedio por Período Presidencial — Colombia<br>'
          '<sup>Valores reales en cada celda | Colores normalizados 0–1 por indicador</sup>',
    height=HEIGHT,
    xaxis=dict(tickangle=-30),
)
fig.show()

print("\n=== Valores reales ===")
print(resumen.to_string())



=== Valores reales ===
                          Crec. PIB (%)  Inflación (%)  Desempleo (%)  IED (% PIB)  Exportac. (% PIB)  Bal. Comercial  Deuda Ext. (% PIB)
presidente                                                                                                                               
ALBERTO LLERAS CAMARGO             5.25           6.26            NaN          NaN              13.63            0.03                 NaN
GUILLERMO LEÓN VALENCIA            4.58          16.94            NaN          NaN              11.51           -0.75                 NaN
CARLOS LLERAS RESTREPO             5.59           7.42            NaN         0.60              12.20           -0.48               32.47
MISAEL PASTRANA BORRERO            6.52          17.70            NaN         0.33              13.67           -0.61               32.14
ALFONSO LÓPEZ MICHELSEN            4.92          23.71            NaN         0.31              16.59            2.86               26.63
JULIO CÉSA

# Análisis de Correlación

In [32]:
## Matriz de Correlación de Pearson — Indicadores Macroeconómicos
CORR_COLS = [
    'total_gdp_million', 'gdp_variation', 'total_gdp_percapita',
    'exports_of_goods_and_services', 'imports_of_goods_and_services',
    'balance_comercial', 'cuenta_corriente', 'inflation_rate',
    'foreign_direct_investment', 'unemployment_rate',
    'external_debt_pct_gdp', 'deuda_publica', 'ingresos_tributarios',
    'poblacion', 'gini', 'life_expectancy_women',
]
CORR_LABELS = [
    'PIB Total', 'Crec. PIB', 'PIB per Cápita',
    'Exportaciones', 'Importaciones',
    'Balance Comercial', 'Cuenta Corriente', 'Inflación',
    'IED Neta', 'Desempleo',
    'Deuda Ext. %PIB', 'Deuda Pública', 'Ing. Tributarios',
    'Población', 'Gini', 'Esp. Vida Mujeres',
]

df_corr = df[CORR_COLS].replace(0, np.nan)
corr_matrix = df_corr.corr(method='pearson').round(2)
corr_matrix.columns = CORR_LABELS
corr_matrix.index   = CORR_LABELS

fig = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=CORR_LABELS, y=CORR_LABELS,
    colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
    text=corr_matrix.values,
    texttemplate='%{text:.2f}',
    textfont=dict(size=8),
    colorbar=dict(title='Pearson r'),
))
fig.update_layout(
    title='Matriz de Correlación de Pearson — Indicadores Macroeconómicos Colombia',
    height=HEIGHT + 100,
    xaxis=dict(tickangle=-45, tickfont=dict(size=9)),
    yaxis=dict(tickfont=dict(size=9)),
)
fig.show()


In [33]:
## Correlación de cada indicador con el Crecimiento del PIB
target = 'gdp_variation'
otros = [c for c in CORR_COLS if c != target]
etiquetas = [CORR_LABELS[CORR_COLS.index(c)] for c in otros]

correlaciones = [
    df_corr[[target, c]].dropna().corr().loc[target, c]
    for c in otros
]

orden = sorted(zip(correlaciones, etiquetas), key=lambda x: x[0])
corr_sorted, lab_sorted = zip(*orden)

colors_bar = ['steelblue' if v >= 0 else 'tomato' for v in corr_sorted]

fig = go.Figure(go.Bar(
    x=list(corr_sorted),
    y=list(lab_sorted),
    orientation='h',
    marker_color=colors_bar,
    text=[f'{v:.2f}' for v in corr_sorted],
    textposition='outside',
))
fig.add_vline(x=0, line_color='black', line_width=1)
fig.update_layout(
    title='Correlación de Pearson de cada Indicador con el Crecimiento del PIB',
    xaxis_title='Coeficiente de Pearson',
    xaxis=dict(range=[-1.1, 1.1]),
    height=HEIGHT,
)
fig.show()


In [34]:
# ── Sección: Inflación mensual vs Tasa de interés (Banco de la República) ────

## Cargar y preparar datos mensuales
df_inflacion = pd.read_csv('data/inflacion.csv')
df_tasa_raw  = pd.read_csv('data/tasa_interes.csv')

# Parsear fechas inflación
df_inflacion['fecha'] = pd.to_datetime(df_inflacion['Periodo(MMM, AAAA)'], format='%Y/%m/%d')
df_inflacion['mes']   = df_inflacion['fecha'].dt.to_period('M')

# Agregar tasa de interés: promedio mensual (desde datos diarios)
df_tasa_raw['fecha'] = pd.to_datetime(df_tasa_raw['Periodo(MMM DD, AAAA)'], format='%Y/%m/%d')
df_tasa_raw['mes']   = df_tasa_raw['fecha'].dt.to_period('M')

df_tasa_mensual = (
    df_tasa_raw.groupby('mes')['Tasa de política monetaria']
    .agg(tasa_promedio='mean', tasa_cierre='last')
    .round(2)
    .reset_index()
)

# Combinar
df_m = pd.merge(
    df_inflacion[['mes', 'fecha', 'Inflación total', 'Meta de inflación']],
    df_tasa_mensual,
    on='mes', how='inner'
).rename(columns={'Inflación total': 'inflacion', 'Meta de inflación': 'meta'})

df_m['year'] = df_m['mes'].dt.year
df_m['mes_str'] = df_m['fecha'].dt.strftime('%b %Y')

# Unir con presidentes
df_m = pd.merge(df_m, df_presidentes.rename(columns={'year':'year'}), on='year', how='left')

print(f"Cobertura: {df_m['fecha'].min().strftime('%b %Y')} → {df_m['fecha'].max().strftime('%b %Y')}")
print(f"Total meses: {len(df_m)}")
df_m.tail(6)


Cobertura: Feb 1998 → Feb 2026
Total meses: 337


,mes,fecha,inflacion,meta,tasa_promedio,tasa_cierre,year,mes_str,presidente
331,2025-09,2025-09-30,4.40,3.0,9.25,9.25,2025,Sep 2025,NaN
332,2025-10,2025-10-31,4.25,3.0,9.25,9.25,2025,Oct 2025,NaN
333,2025-11,2025-11-30,4.01,3.0,9.25,9.25,2025,Nov 2025,NaN
334,2025-12,2025-12-31,3.95,3.0,9.25,9.25,2025,Dec 2025,NaN
335,2026-01,2026-01-31,5.35,3.0,9.31,10.25,2026,Jan 2026,NaN
336,2026-02,2026-02-28,5.29,3.0,10.25,10.25,2026,Feb 2026,NaN


In [35]:
## Serie temporal: Inflación mensual y Tasa de interés (1998–2026)
from plotly.subplots import make_subplots

df_plot = df_m[df_m['fecha'].dt.year >= 1998].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Inflación Mensual IPC (%)',
                                    'Tasa de Política Monetaria (%)'))

# Inflación — coloreada por presidente
for pres in PRESIDENTES_ORDEN:
    sub = df_plot[df_plot['presidente'] == pres]
    if sub.empty:
        continue
    fig.add_trace(go.Scatter(
        x=sub['fecha'], y=sub['inflacion'],
        mode='lines', name=pres,
        line=dict(color=PRES_COLOR_MAP[pres], width=1.5),
        legendgroup=pres, showlegend=True,
    ), row=1, col=1)

# Meta de inflación
df_meta = df_plot[df_plot['meta'].notna()]
fig.add_trace(go.Scatter(
    x=df_meta['fecha'], y=df_meta['meta'],
    mode='lines', name='Meta BanRep (3%)',
    line=dict(color='black', dash='dot', width=1.2),
    legendgroup='meta',
), row=1, col=1)

# Tasa de interés — misma paleta por presidente
for pres in PRESIDENTES_ORDEN:
    sub = df_plot[df_plot['presidente'] == pres]
    if sub.empty:
        continue
    fig.add_trace(go.Scatter(
        x=sub['fecha'], y=sub['tasa_promedio'],
        mode='lines', name=pres,
        line=dict(color=PRES_COLOR_MAP[pres], width=1.5),
        legendgroup=pres, showlegend=False,
    ), row=2, col=1)

fig.update_layout(
    title='Inflación Mensual e Tasa de Política Monetaria — Colombia (1998–2026)<br>'
          '<sup>Coloreado por período presidencial</sup>',
    height=HEIGHT,
)
fig.update_yaxes(title_text='Inflación (%)', row=1, col=1)
fig.update_yaxes(title_text='Tasa (%)', row=2, col=1)
fig.show()


In [36]:
## Análisis bivariable mes a mes: Inflación vs Tasa de interés (1998–2026)

df_biv = df_m[df_m['fecha'].dt.year >= 1998].dropna(subset=['inflacion','tasa_promedio'])

fig = go.Figure()

for pres in PRESIDENTES_ORDEN:
    sub = df_biv[df_biv['presidente'] == pres]
    if sub.empty:
        continue
    fig.add_trace(go.Scatter(
        x=sub['tasa_promedio'],
        y=sub['inflacion'],
        mode='markers',
        name=pres,
        marker=dict(
            color=PRES_COLOR_MAP[pres],
            size=7,
            opacity=0.75,
            line=dict(width=0.5, color='white'),
        ),
        text=sub['mes_str'],
        hovertemplate='<b>%{text}</b><br>Tasa: %{x:.2f}%<br>Inflación: %{y:.2f}%<extra>%{fullData.name}</extra>',
        legendgroup=pres,
    ))

# Línea de tendencia global
x_all = df_biv['tasa_promedio'].values
y_all = df_biv['inflacion'].values
coeffs = np.polyfit(x_all, y_all, 1)
x_line = np.linspace(x_all.min(), x_all.max(), 200)
r = np.corrcoef(x_all, y_all)[0, 1]
fig.add_trace(go.Scatter(
    x=x_line, y=np.polyval(coeffs, x_line),
    mode='lines',
    name=f'Tendencia lineal (r = {r:.2f})',
    line=dict(color='black', dash='dash', width=2),
))

# Línea tasa real = 0 (tasa = inflación)
lim = max(x_all.max(), y_all.max()) * 1.08
fig.add_trace(go.Scatter(
    x=[0, lim], y=[0, lim],
    mode='lines',
    name='Tasa real = 0% (tasa = inflación)',
    line=dict(color='gray', dash='dot', width=1.2),
))

fig.update_layout(
    title='Inflación Mensual vs Tasa de Política Monetaria — Colombia (1998–2026)<br>'
          '<sup>Cada punto = un mes | Línea gris: tasa real = 0% | '
          f'r = {r:.2f} (correlación de Pearson global)</sup>',
    xaxis_title='Tasa de política monetaria promedio mensual (%)',
    yaxis_title='Inflación mensual IPC (%)',
    height=HEIGHT,
    legend=dict(font=dict(size=9)),
)
fig.show()


In [37]:
## Tasa de interés real mensual: tasa nominal − inflación acumulada 12 meses

# Calcular inflación acumulada 12 meses
df_m_sorted = df_m.sort_values('fecha').copy()
df_m_sorted['inflacion_anual_acum'] = (
    (1 + df_m_sorted['inflacion'] / 100)
    .rolling(12)
    .apply(lambda x: x.prod() - 1, raw=True) * 100
).round(2)

df_m_sorted['tasa_real'] = (df_m_sorted['tasa_promedio'] - df_m_sorted['inflacion_anual_acum']).round(2)

df_real = df_m_sorted[df_m_sorted['fecha'].dt.year >= 1998].dropna(subset=['tasa_real'])

fig = go.Figure()

for pres in PRESIDENTES_ORDEN:
    sub = df_real[df_real['presidente'] == pres]
    if sub.empty:
        continue
    fig.add_trace(go.Scatter(
        x=sub['fecha'], y=sub['tasa_real'],
        mode='lines', name=pres,
        line=dict(color=PRES_COLOR_MAP[pres], width=1.5),
        legendgroup=pres,
        fill='tozeroy',
        fillcolor=PRES_COLOR_MAP[pres].replace('#','rgba(').replace('b4','b4,0.08)')             if False else 'rgba(0,0,0,0)',
    ))

fig.add_hline(y=0, line_color='black', line_width=1.2,
              annotation_text='Tasa real = 0%', annotation_position='top left')

fig.update_layout(
    title='Tasa de Interés Real Mensual — Colombia (1998–2026)<br>'
          '<sup>Tasa nominal − Inflación acumulada 12 meses | '
          'Zona positiva: política restrictiva | Zona negativa: política expansiva</sup>',
    xaxis_title='Fecha',
    yaxis_title='Tasa real (%)',
    height=HEIGHT,
    legend=dict(font=dict(size=9)),
)
fig.show()
